# Download and process WHACS offshore waves

Download WHACS bulk and partition spectra, remap them onto metamodel goals, and
write the processed zarr used by dynamic forcing and hindcast reconstruction.

**Prerequisites:** ``04_Select_Goals_by_Hindcasts.ipynb`` (``<Case>_Goals.json``).
**Next:** ``07b_Satellite_Calibration.ipynb`` validates (and calibrates) this
hindcast against satellite altimetry.

## Configuration


In [1]:
import numpy as np
import xarray as xr

from utils.config import case_name, get_offshore_wave, load_goals, resolve

whacs_data = xr.open_dataset(
    resolve(f"inputs/WHACS/processed/{case_name()}_WHACS_seapoint.zarr")
).load()
whacs_data = whacs_data.assign({"tp": 1.0 / whacs_data["fp"]})
whacs_data["tp"].attrs = {
    "long_name": "Wave peak period",
    "units": "s",
}
whacs_data

<xarray.Dataset> Size: 975MB
Dimensions:    (time: 394464, seapoint: 14)
Coordinates:
  * time       (time) datetime64[ns] 3MB 1979-01-01 ... 2023-12-31T23:00:00
  * seapoint   (seapoint) int32 56B 1903180 1903191 1903201 ... 1911543 1911562
Data variables: (12/23)
    dir        (time, seapoint) float64 44MB 128.9 127.5 127.9 ... 191.7 186.1
    fp         (time, seapoint) float64 44MB 0.19 0.194 0.202 ... 0.098 0.097
    hs         (time, seapoint) float64 44MB 0.904 1.054 1.132 ... 0.69 0.638
    latitude   (time, seapoint) float32 22MB 33.12 33.12 33.12 ... 36.06 36.44
    longitude  (time, seapoint) float32 22MB -79.0 -78.31 ... -74.88 -74.81
    pdir0      (time, seapoint) float64 44MB nan 128.6 129.3 ... 237.5 229.2
    ...         ...
    ptp0       (time, seapoint) float64 44MB nan 5.15 4.95 ... 6.85 3.01 3.08
    ptp1       (time, seapoint) float64 44MB 5.25 7.62 7.62 ... 10.36 6.99 7.11
    ptp2       (time, seapoint) float64 44MB 9.4 9.02 8.83 ... 3.69 10.4 10.45
    ptp3       (time, seapoint) float64 44MB 12.77 12.68 12.61 ... 3.52 3.24
    spr        (time, seapoint) float64 44MB 31.8 32.6 33.8 ... 55.4 65.0 66.2
    tp         (time, seapoint) float64 44MB 5.263 5.155 4.95 ... 10.2 10.31
Attributes:
    source:   WHACS global archive /lustre/geocean/DATA/CSIRO/WHACS/global-smc
    case:     NorthCarolina

In [2]:
# Offshore node per goal, matched by lon/lat to the goals JSON snap points
# (hindcast.whacs.longitude/latitude). Some hindcasts are gridded rather
# than point-organized, so matching by position is more robust than by id.
goals = load_goals()
seapoint_lon = whacs_data["longitude"].isel(time=-1)
seapoint_lat = whacs_data["latitude"].isel(time=-1)
ordered_seapoints_for_goals = []
for goal in goals.values():
    snap = goal["hindcast"]["whacs"]
    distance = np.hypot(
        seapoint_lon - snap["longitude"], seapoint_lat - snap["latitude"]
    )
    ordered_seapoints_for_goals.append(
        int(whacs_data["seapoint"][int(distance.argmin())])
    )

whacs_data = whacs_data.sel(seapoint=ordered_seapoints_for_goals)[
    [
        "hs",
        "tp",
        "dir",
        "spr",
        "phs0",
        "ptp0",
        "pdir0",
        "pspr0",
        "phs1",
        "ptp1",
        "pdir1",
        "pspr1",
        "phs2",
        "ptp2",
        "pdir2",
        "pspr2",
        "phs3",
        "ptp3",
        "pdir3",
        "pspr3",
    ]
].rename({"seapoint": "goal"})
whacs_data["goal"] = [int(k) for k in goals]
whacs_data = whacs_data.assign_coords(
    {"seapoint": ("goal", ordered_seapoints_for_goals)}
)

Bulk `tp = 1 / fp` explodes when the spectral peak falls below the WHACS
frequency grid (hours with Tp of 50-250 s). Those take the peak period of the
most energetic partition (`clean_peak_periods`); partition periods above
`max_tp` become NaN.

In [3]:
from bluemath_tk.waves.hywaves.offshore import clean_peak_periods, scale_partitions_to_bulk

max_tp = 25.0  # s, largest valid peak period
whacs_data = clean_peak_periods(whacs_data, max_tp=max_tp)
print(f"bulk tp replaced: {whacs_data.attrs['tp_replaced']}, partition tp removed: {whacs_data.attrs['ptp_removed']}")

bulk tp replaced: 0, partition tp removed: 0


**Partition energy.** WHACS keeps the wind sea and three swells; where the
sea state has more systems (multimodal tropical seas) the partitions hold less
energy than the bulk spectrum, and the spectral reconstruction (a sum of
partitions) would start low. `scale_partitions_to_bulk` gives each hour's
partitions the energy they miss, in proportion to theirs (`phs` x
`hs / sqrt(sum phs²)`; periods, directions and spreads kept). The share they
held before, per goal, is printed below. The satellite calibration (07b) scales
the bulk and the partitions by the same factors, so it keeps the closure.

In [4]:
energy = sum(whacs_data[f"phs{i}"].fillna(0.0) ** 2 for i in range(4))
captured = (energy.sum("time") / (whacs_data["hs"] ** 2).sum("time")).to_series()
print("partition energy / bulk energy per goal, before scaling:")
print(captured.round(3).to_string())

whacs_data = scale_partitions_to_bulk(whacs_data)
whacs_data.to_zarr(resolve(get_offshore_wave("WHACS")["data_path"]), mode="w")

partition energy / bulk energy per goal, before scaling:
goal
1     0.997
2     0.996
3     0.996
4     0.996
5     0.997
6     0.997
7     0.997
8     0.997
9     0.997
10    0.998
11    0.998
12    0.998
13    0.998
14    0.998


In [5]:
xr.open_zarr(resolve(get_offshore_wave("WHACS")["data_path"])).load()

<xarray.Dataset> Size: 887MB
Dimensions:   (time: 394464, goal: 14)
Coordinates:
  * time      (time) datetime64[ns] 3MB 1979-01-01 ... 2023-12-31T23:00:00
  * goal      (goal) int64 112B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    seapoint  (goal) int64 112B 1904519 1911562 1911543 ... 1903191 1903180
Data variables: (12/20)
    dir       (time, goal) float64 44MB 68.4 66.9 64.4 ... 240.8 232.4 211.9
    hs        (time, goal) float64 44MB 0.476 0.532 0.576 ... 1.342 1.32 1.026
    pdir0     (time, goal) float64 44MB nan nan 155.5 ... 242.6 234.1 214.6
    pdir1     (time, goal) float64 44MB 61.3 57.9 56.2 53.5 ... 97.3 106.3 110.6
    pdir2     (time, goal) float64 44MB 68.0 68.2 60.5 62.6 ... 134.4 nan nan
    pdir3     (time, goal) float64 44MB 77.8 84.9 65.9 74.8 ... nan nan nan nan
    ...        ...
    ptp0      (time, goal) float64 44MB nan nan 2.55 3.26 ... 5.12 4.94 4.57
    ptp1      (time, goal) float64 44MB 7.26 7.37 7.14 6.8 ... 10.37 10.39 10.83
    ptp2      (time, goal) float64 44MB 12.44 12.39 12.39 12.4 ... 9.34 nan nan
    ptp3      (time, goal) float64 44MB 16.84 16.84 16.8 16.74 ... nan nan nan
    spr       (time, goal) float64 44MB 22.0 21.6 27.6 50.2 ... 34.9 31.6 30.6
    tp        (time, goal) float64 44MB 16.67 16.67 16.67 ... 5.128 4.95 4.566
Attributes:
    source:                  WHACS global archive /lustre/geocean/DATA/CSIRO/...
    case:                    NorthCarolina
    tp_replaced:             0
    ptp_removed:             0
    partition_energy_share:  0.9971